## 01 — Diagnosing ambiguous XDec profiles (TCGA-PAAD, n=150, k=9)

* **Aim:**
In the initial run, Profile_7 correlated broadly with reference expression across nearly all cell types but was the dominant profile in only one sample. This pattern raises concern about a possible NNLS estimation artifact, although it is not conclusive. To investigate its cause, I am aligning the inputs and parameters with the paper—its 149-sample tumor cohort, k=9, Table S3 probes, and reference preprocessing—and testing whether the profile persists as each discrepancy is addressed. Section 4 additionally tests removing that single originally-flagged sample directly, which also brings the cohort to the paper's reported n=149. Section 5 adapts XDec-CHI's own sliding-window method (originally used for gene-gene interaction discovery) into a diagnostic: refitting Stage 1/2 inside overlapping 40-sample windows ranked by enrichment for Profile_7, to test whether its signature sharpens with enrichment (real compartment) or stays flat (artifact).

**Possible reasons for the apparent artifact**

* **Paper settings:** 149 samples · **k=9** · 283 genes (Table S3)
* **This run:** 150-sample cohort · Run A uses 282 recovered Table S3 symbols (247 usable for correlation)
* Paper reports **bulk tumors only** — no confirmed normal-tissue controls in the 149-sample cohort (barcode-level check not yet done). See the [paper, Results and STAR Methods](https://pmc.ncbi.nlm.nih.gov/articles/PMC9579012/).
* These checks narrow possible explanations only — they do **not** establish that the ambiguous profile is an artifact, nor reproduce the paper's two cancer states

**By resolution status**

✅ **Ruled out**

| Reason | Evidence |
|---|---|
| Different **k** | Paper and local runs both use **k=9** |
| Missing exponential transform | Local CLI already applies per-gene transform to bulk + refs |

⚠️ **Partially addressed**

| Reason | Evidence |
|---|---|
| Unfiltered bulk samples | Ambiguity persists at **n=150**; not the exact 149-sample cohort |
| Selected probes | **Table S3 genes** (Run A): max breadth 10→5; no-housekeeping (Run B): →4 |
| Reference preprocessing | 5-cell pseudo-bulk (Run D) retains breadth 7/10; depth-ranking unavailable |
| Real vs. artifactual minority compartment | Section 5 sliding-window check: breadth/margin/zero-floor counts do **not** sharpen with enrichment, and matched-profile identity to baseline Profile_7 is weak (*r*=0.30–0.51) across windows — consistent with, not proof of, an artifact |

❓ **Unresolved**

| Reason | Evidence |
|---|---|
| Reference composition / expression source | Local ref/bulk not verified equivalent to paper's inputs |
| Initialization & convergence | Single seed; nonconvex joint factorization |
| Implementation differences | Python port not validated end-to-end vs. paper's code |

**Priority:** fix cohorts at **k=9** → evaluate matched profiles on shared genes across seeds → test epithelial markers + independent tumor purity before labeling unmatched profiles as cancer states.

In [ ]:
from pathlib import Path
import os
import re
import subprocess

import numpy as np
import pandas as pd

# Find progress/ from either the notebook directory or progress/ itself.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "tools/XDec_SM/python/xdec.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start from progress/ or one of its subdirectories.")
os.chdir(ROOT)
XDEC_DIR = "data/deconvolution/xdec"
RBASE_RSCRIPT = str(Path.home() / ".pixi/envs/r-base/bin/Rscript")
PYTHON_PIXI = str(Path.home() / ".pixi/envs/python/bin/python3")
XDEC_SCRIPT = "tools/XDec_SM/python/xdec.py"
RUN_PIPELINE = False  # Opt in to generating missing outputs; existing outputs are reused.
CORR_THRESHOLD = 0.3


def run_checked(command):
    if not RUN_PIPELINE:
        raise FileNotFoundError("Missing pipeline output. Set RUN_PIPELINE=True to generate it.")
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {command}\n"
                           f"{result.stdout}\n{result.stderr}")
    print(result.stdout)
    return result


def read_genes(path):
    genes = Path(path).read_text().splitlines()
    return list(dict.fromkeys(g.strip() for g in genes if g.strip()))


def validate_outputs(props, corr, means):
    for name, frame in {"proportions": props, "correlation": corr, "means": means}.items():
        if not frame.index.is_unique or not frame.columns.is_unique:
            raise ValueError(f"Duplicate labels in {name}")
        if not np.isfinite(frame.to_numpy()).all():
            raise ValueError(f"Nonfinite values in {name}")
    if set(props.columns) != set(corr.columns) or set(props.columns) != set(means.columns):
        raise ValueError("Profile labels disagree between Stage 1/2 outputs")
    if props.shape != (150, 9) or not props.index.equals(restricted_columns):
        raise ValueError("Proportions do not match the ordered 150-sample cohort at k=9")
    if (props.to_numpy() < 0).any() or (means.to_numpy() < 0).any():
        raise ValueError("Negative proportions or Stage 2 means")
    if not np.allclose(props.sum(axis=1), 1, atol=1e-5):
        raise ValueError("Proportions do not sum to one")
    if (np.abs(corr.to_numpy()) > 1 + 1e-8).any():
        raise ValueError("Correlation outside [-1, 1]")

### 1. Unfiltered samples

The [pipeline README](../scripts/README.md) describes the original download and QC workflow. The detailed [sample-filtering rationale](sample_filtering.md) explains why the full 182-sample input differs from the 150-sample PanCanAtlas cohort used here and XDec-CHI's reported 149 samples.

Use the local `TCGAbiolinks::TCGAquery_subtype(tumor="paad")` export to restrict the cohort. Match at the 16-character sample/vial level and reject duplicate matches or missing samples. Validate an existing subset against the source counts before reuse.


In [ ]:
subtype_dir = os.path.join(XDEC_DIR, "subtypes")
subtype_tsv = os.path.join(subtype_dir, "paad_panCanAtlas_subtypes.tsv")
r_script_path = os.path.join(subtype_dir, "pull_paad_subtypes.R")

if not os.path.exists(subtype_tsv):
    os.makedirs(subtype_dir, exist_ok=True)
    result = run_checked([RBASE_RSCRIPT, r_script_path])
assert os.path.exists(subtype_tsv)

subtypes = pd.read_csv(subtype_tsv, sep="\t")
print(subtypes.shape)
subtypes.head(3)

In [ ]:
restricted_bulk_path = os.path.join(XDEC_DIR, "TCGA-PAAD_counts_raw_n150.tsv")
counts = pd.read_csv("data/processed/TCGA-PAAD_counts_raw.tsv", sep="\t", index_col=0)
cohort = subtypes["tumor_sample_id"].astype(str).str.strip()
if len(cohort) != 150 or not cohort.is_unique or not cohort.str.len().eq(16).all():
    raise ValueError("Expected 150 unique 16-character subtype barcodes")
if not counts.columns.is_unique or not counts.index.is_unique:
    raise ValueError("Duplicate bulk sample or gene labels")
barcodes = counts.columns.str[:16]
keep = barcodes.isin(cohort)
matched = barcodes[keep]
if not matched.is_unique or set(matched) != set(cohort):
    raise ValueError("Bulk-to-cohort matching is missing samples or is not one-to-one")
restricted = counts.loc[:, keep]
restricted_columns = restricted.columns
if Path(restricted_bulk_path).exists():
    cached = pd.read_csv(restricted_bulk_path, sep="\t", index_col=0)
    pd.testing.assert_frame_equal(cached, restricted, check_dtype=False)
else:
    if not RUN_PIPELINE:
        raise FileNotFoundError("Missing cohort subset; set RUN_PIPELINE=True to create it")
    restricted.to_csv(restricted_bulk_path, sep="\t")
print(f"Validated {restricted.shape[0]} genes × {restricted.shape[1]} samples")
del counts, restricted

Use the original reference and Stage 0 probes, k=9, and the CLI's default seed (12345). Existing outputs are reused, not regenerated. Completeness and matrix consistency are checked below; these checks cannot establish the provenance of legacy cached fits. Changing inputs requires new output directories or deliberate cache invalidation.

In [ ]:
stage1_dir = os.path.join(XDEC_DIR, "stage1_with_refs_k9_n150")
os.makedirs(stage1_dir, exist_ok=True)
proportions_path = os.path.join(stage1_dir, "proportions.tsv")

if not all(Path(stage1_dir, name).exists() for name in
           ("proportions.tsv", "cell_type_profiles.tsv", "correlation.tsv")):
    result = run_checked(
        [PYTHON_PIXI, XDEC_SCRIPT, "stage1",
         "--bulk", restricted_bulk_path,
         "--probes", os.path.join(XDEC_DIR, "stage0", "selected_probes.txt"),
         "--refs", os.path.join(XDEC_DIR, "refs.tsv"),
         "--ref-metadata", os.path.join(XDEC_DIR, "ref_metadata.tsv"),
         "--k", "9",
         "--outdir", stage1_dir],
    )

proportions = pd.read_csv(proportions_path, sep="\t", index_col=0)
proportions.shape

In [ ]:
stage2_dir = os.path.join(XDEC_DIR, "stage2_with_refs_k9_n150")
os.makedirs(stage2_dir, exist_ok=True)
means_path = os.path.join(stage2_dir, "stage2_means.tsv")

if not all(Path(stage2_dir, name).exists() for name in
           ("stage2_means.tsv", "stage2_stderrs.tsv")):
    result = run_checked(
        [PYTHON_PIXI, XDEC_SCRIPT, "stage2",
         "--bulk-counts", restricted_bulk_path,
         "--proportions", proportions_path,
         "--outdir", stage2_dir],
    )

stage2_means = pd.read_csv(means_path, sep="\t", index_col=0)
stage2_means.shape

Report three complementary summaries, without treating them as independent proof of an artifact:

- **Dominance:** number of samples where a profile has the largest fitted proportion; this is not prevalence or a count above 50%.
- **Breadth:** reference types with mean single-cell Spearman correlation >0.3, as calculated by the local CLI over the run's usable probes. Zero breadth means no correlation exceeds this threshold, not zero correlation.
- **Boundary estimates:** genes above the candidate's 99th expression percentile with an exact zero in another profile. The partner maximizes this count; it is not necessarily the most biologically similar profile. These are descriptive contrasts, not differential-expression tests.

Profile_7 is inspected only in the baseline fit. Cross-run comparisons below select the broadest profile separately and do not assume matching profile identities.

In [ ]:
corr = pd.read_csv(os.path.join(stage1_dir, "correlation.tsv"), sep="\t", index_col=0)
validate_outputs(proportions, corr, stage2_means)
corr.shape

In [ ]:
dominant_counts = proportions.idxmax(axis=1).value_counts().reindex(proportions.columns, fill_value=0)
dominant_counts

In [ ]:
broad_corr = (corr > CORR_THRESHOLD).sum(axis=0).sort_values(ascending=False)
broad_corr.rename("n_reference_types_with_r_gt_0.3")

In [ ]:
profile7_means = stage2_means["Profile_7"]

zero_floor_rows = []
for other in stage2_means.columns:
    if other == "Profile_7":
        continue
    other_vals = stage2_means[other]
    mask = (other_vals == 0) & (profile7_means > profile7_means.quantile(0.99))
    zero_floor_rows.append({"partner_profile": other, "n_genes_zero_floor": int(mask.sum())})
zero_floor_df = pd.DataFrame(zero_floor_rows).sort_values("n_genes_zero_floor", ascending=False)
zero_floor_df


In [ ]:
best_partner = zero_floor_df.iloc[0]["partner_profile"]
mask = (stage2_means[best_partner] == 0) & (profile7_means > profile7_means.quantile(0.99))
top_genes = profile7_means[mask].sort_values(ascending=False).head(20)
print(f"top genes where Profile_7 is in its top 1% of expression and {best_partner} == 0.0:")
print(top_genes)

ribo_pattern = re.compile(r"^(RPL|RPS|MRPL|MRPS|EEF|RPLP)")
frac_ribo = top_genes.index.str.match(ribo_pattern).mean() if len(top_genes) else np.nan
print(f"\nfraction of these top 20 genes matching a ribosomal/translation gene-symbol pattern: {frac_ribo:.2f}")

**Interpretation:** broad reference similarity and boundary estimates warrant further checks, but neither establishes a spurious component. The top boundary genes include epithelial/secretory as well as housekeeping-related symbols, so genuine epithelial signal remains plausible. Low dominance alone is compatible with a real minority compartment.

The earlier n=182/k=8 result and this n=150/k=9 result change both cohort and k. They cannot isolate a cohort effect; that requires comparing cohorts at the same k and assessing multiple initializations.

### 2. Selected probes (a confirmed partial cause)

Plot `log1p` of the exported reference values without per-gene standardization. This shows absolute expression patterns on the export's scale, not raw sequencing counts. The reference-only plot is unaffected by cohort restriction. Broad expression does not by itself imply lack of cell-type information; relative differences may remain informative.

The probe interventions below confirm that selected probes contribute to the observed correlation breadth. This is a partial explanation of the diagnostic signature; it does not establish that an entire profile is artifactual.


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

mpl.rcParams.update({"font.family": "sans-serif", "font.size": 8, "axes.linewidth": 0.6, "pdf.fonttype": 42})

FIG_DIR = os.path.join(XDEC_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Single-hue sequential blue ramp (magnitude encoding), light -> dark.
SEQ_BLUE_STEPS = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
                  "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
seq_blue = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE_STEPS, N=256)


In [ ]:
refs = pd.read_csv(os.path.join(XDEC_DIR, "refs.tsv"), sep="\t", index_col=0)
ref_meta = pd.read_csv(os.path.join(XDEC_DIR, "ref_metadata.tsv"), sep="\t", index_col=0)
probe_list = read_genes(os.path.join(XDEC_DIR, "stage0", "selected_probes.txt"))
valid_probes = [g for g in probe_list if g in refs.index]
print(f"{len(valid_probes)}/{len(probe_list)} probes present in refs.tsv")

if not refs.index.is_unique or not refs.columns.is_unique or not ref_meta.index.is_unique:
    raise ValueError("Duplicate reference or metadata labels")
if not np.isfinite(refs.to_numpy()).all() or (refs.to_numpy() < 0).any():
    raise ValueError("Reference must contain finite nonnegative expression")
ref_cell_types = ref_meta.loc[refs.columns, "Cell_Type"]
if ref_cell_types.isna().any() or len(valid_probes) < 2:
    raise ValueError("Missing cell types or insufficient shared probes")
# Match the biological order and annotation colors in 02_visualization.ipynb.
cell_type_order = [
    "pancreatic acinar cell", "pancreatic ductal cell", "enteroendocrine cell",
    "pancreatic stellate cell", "fibroblast", "blood vessel endothelial cell", "neural cell",
    "T cell", "lymphocyte of B lineage", "myeloid leukocyte",
]
if set(cell_type_order) != set(ref_cell_types.unique()):
    raise ValueError("Cell-type order must cover the reference metadata")
cell_palette = dict(zip(cell_type_order, sns.color_palette("tab10", len(cell_type_order))))
cell_order = ref_cell_types.map({t: i for i, t in enumerate(cell_type_order)}).sort_values(kind="stable").index
expr = np.log1p(refs.loc[valid_probes, cell_order])  # genes × cells; retain expression scale
col_colors = ref_cell_types.loc[cell_order].map(cell_palette)

g = sns.clustermap(
    expr, row_cluster=True, col_cluster=False,
    col_colors=col_colors.to_numpy(),
    cmap=seq_blue, vmin=0, vmax=float(np.percentile(expr.values, 99)),
    xticklabels=False, yticklabels=False,
    figsize=(10, 7), cbar_kws={"label": "log1p(expression)"},
    dendrogram_ratio=(0.08, 0.02),
)
g.ax_heatmap.set_xlabel(f"{len(cell_order)} reference single cells (sorted by cell type, not clustered)")
g.ax_heatmap.set_ylabel(f"{len(valid_probes)} Stage-0 selected probes (hierarchically clustered)")
handles = [plt.Rectangle((0, 0), 1, 1, color=cell_palette[t]) for t in cell_type_order]
legend = g.ax_heatmap.legend(
    handles, cell_type_order, title="Cell type", bbox_to_anchor=(1.05, 1.02),
    loc="upper left", fontsize=7, title_fontsize=8, frameon=False,
)
g.fig.suptitle("Peng et al. reference expression over the selected probes\n"
               "log1p expression; cells sorted by biological compartment", y=1.03, fontsize=10)

# Align the expression colorbar beneath the cell-type legend, as in Figure 1 of notebook 02.
g.cax.yaxis.label.set_fontsize(7)
g.fig.canvas.draw()
legend_bbox = legend.get_window_extent(g.fig.canvas.get_renderer()).transformed(g.fig.transFigure.inverted())
g.cax.set_position([legend_bbox.x0, legend_bbox.y0 - 0.18, 0.025, 0.15])

expression_fig_dir = Path("paper_duplication/figures")
expression_fig_dir.mkdir(parents=True, exist_ok=True)
fig1_path = expression_fig_dir / "fig_reference_expression_clustermap_n150_k9.png"
g.savefig(fig1_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(g.fig)
print("saved", fig1_path)


**Interpretation:** dark columns identify broadly abundant probes on this scale. The comparison below ranks them quantitatively; abundance and cell-type specificity are distinct properties.

Compare the local Stage 0 list with the 282 unique symbols in the extracted Table S3 file. The local extraction is incomplete relative to the reported 283 entries; do not infer the missing symbol from a date-like string without checking the original table.

The symbol-prefix pattern below is a reproducible heuristic for ribosomal/translation and other broadly expressed gene families. It is not a curated housekeeping annotation, and matching genes may still carry biological signal.

In [ ]:
paper_genes_path = os.path.join(XDEC_DIR, "gene_lists", "xdec_chi_table_s3_genes.txt")
paper_genes = read_genes(paper_genes_path)
paper_set = set(paper_genes)
our_set = set(probe_list)

overlap = our_set & paper_set
print(f"Our {len(our_set)} Stage-0 probes vs. the paper's {len(paper_set)} Table S3 genes:")
print(f"  overlap: {len(overlap)} genes "
      f"({len(overlap) / len(our_set):.1%} of ours, {len(overlap) / len(paper_set):.1%} of theirs)")

In [ ]:
ribo_hk_pattern = re.compile(r"^(RPL|RPS|MRPL|MRPS|EEF|RPLP|TMSB|B2M$|ACTB$|GAPDH$|TUBB|TUBA|HSP|FTL$|FTH1$)")
our_hk = sorted(g for g in our_set if ribo_hk_pattern.match(g))
paper_hk = sorted(g for g in paper_set if ribo_hk_pattern.match(g))

print(f"Our housekeeping/ribosomal-pattern genes: {len(our_hk)}/{len(our_set)} ({len(our_hk)/len(our_set):.1%})")
print(our_hk)
print(f"\nPaper's housekeeping/ribosomal-pattern genes: {len(paper_hk)}/{len(paper_set)} "
      f"({len(paper_hk)/len(paper_set):.1%})")
print(paper_hk)


In [ ]:
# Rank by the minimum log1p(type mean); this is not mean(log1p) as shown in Figure 1.
# across the 10 cell types (reusing refs/ref_cell_types/valid_probes from Figure 1 above).
by_type_mean = refs.loc[valid_probes].T.groupby(ref_cell_types).mean().T
log_by_type = np.log1p(by_type_mean)
high_everywhere = log_by_type.min(axis=1).sort_values(ascending=False).head(30)

in_paper = high_everywhere.index.isin(paper_set)
print(f"Of Figure 1's top 30 'high everywhere' genes, {in_paper.sum()}/30 are in the paper's Table S3 list:")
print(pd.DataFrame({"min_log1p_across_types": high_everywhere.round(2), "in_paper_list": in_paper}))

**Interpretation:** report overlap with both denominators, since the lists differ in size. Prefix-match fractions describe list composition, not contamination rates. The top-30 ranking uses the minimum of `log1p(type mean)`, whereas the heatmap displays individual-cell `log1p` values. These summaries are related but need not identify exactly the same genes.

The local Stage 0 code uses one-vs-rest Welch tests and mean differences without an explicit housekeeping exclusion. This permits broadly expressed genes with relative cell-type differences; it does not establish that they are uninformative or caused an ambiguous profile.

A uses the extracted Table S3 symbols. B removes prefix-matching genes from the original probes. Both retain the same cohort, reference, k, and CLI seed. A changes many genes simultaneously; B changes both composition and list size. Neither isolates a universal housekeeping effect.

Correlation is computed over each run's own usable probes, so breadth changes reflect both the fitted profiles and a changed evaluation gene set. A stronger comparison would evaluate matched profiles on a common held-out gene set and repeat fits across seeds.

In [ ]:
no_hk_probes_path = os.path.join(XDEC_DIR, "stage0", "selected_probes_no_housekeeping.txt")
filtered = [g for g in probe_list if g not in set(our_hk)]
if Path(no_hk_probes_path).exists():
    if read_genes(no_hk_probes_path) != filtered:
        raise ValueError("Cached filtered probes disagree with the current pattern; use new run directories")
else:
    if not RUN_PIPELINE:
        raise FileNotFoundError("Missing filtered probe list; set RUN_PIPELINE=True to create it")
    Path(no_hk_probes_path).write_text("\n".join(filtered) + "\n")

runs = {
    "A_paperGenes": {"probes": paper_genes_path, "outdir_suffix": "paperGenes"},
    "B_noHK": {"probes": no_hk_probes_path, "outdir_suffix": "noHK"},
}

for label, cfg in runs.items():
    s1_dir = os.path.join(XDEC_DIR, f"stage1_with_refs_k9_n150_{cfg['outdir_suffix']}")
    s2_dir = os.path.join(XDEC_DIR, f"stage2_with_refs_k9_n150_{cfg['outdir_suffix']}")
    os.makedirs(s1_dir, exist_ok=True)
    os.makedirs(s2_dir, exist_ok=True)
    prop_path = os.path.join(s1_dir, "proportions.tsv")
    means_path2 = os.path.join(s2_dir, "stage2_means.tsv")

    if not all(Path(s1_dir, name).exists() for name in
               ("proportions.tsv", "cell_type_profiles.tsv", "correlation.tsv")):
        result = run_checked(
            [PYTHON_PIXI, XDEC_SCRIPT, "stage1",
             "--bulk", restricted_bulk_path,
             "--probes", cfg["probes"],
             "--refs", os.path.join(XDEC_DIR, "refs.tsv"),
             "--ref-metadata", os.path.join(XDEC_DIR, "ref_metadata.tsv"),
             "--k", "9",
             "--outdir", s1_dir],
            )
        print(label, "stage1:", result.returncode)

    if not all(Path(s2_dir, name).exists() for name in
               ("stage2_means.tsv", "stage2_stderrs.tsv")):
        result = run_checked(
            [PYTHON_PIXI, XDEC_SCRIPT, "stage2",
             "--bulk-counts", restricted_bulk_path,
             "--proportions", prop_path,
             "--outdir", s2_dir],
            )
        print(label, "stage2:", result.returncode)

    cfg["stage1_dir"] = s1_dir
    cfg["stage2_dir"] = s2_dir

print("done")

In [ ]:
def diagnose(label, stage1_dir, stage2_dir, probes_path):
    props_r = pd.read_csv(os.path.join(stage1_dir, "proportions.tsv"), sep="\t", index_col=0)
    corr_r = pd.read_csv(os.path.join(stage1_dir, "correlation.tsv"), sep="\t", index_col=0)
    means_r = pd.read_csv(os.path.join(stage2_dir, "stage2_means.tsv"), sep="\t", index_col=0)

    validate_outputs(props_r, corr_r, means_r)
    profiles = pd.read_csv(Path(stage1_dir) / "cell_type_profiles.tsv", sep="\t", index_col=0)
    usable = set(read_genes(probes_path)) & set(profiles.index) & set(refs.index[refs.sum(axis=1).gt(0)])
    dom = props_r.idxmax(axis=1).value_counts()
    breadth = (corr_r > CORR_THRESHOLD).sum(axis=0).sort_values(ascending=False, kind="stable")
    candidate = breadth.idxmax()

    cand_means = means_r[candidate]
    zf_rows = []
    for other in means_r.columns:
        if other == candidate:
            continue
        mask = (means_r[other] == 0) & (cand_means > cand_means.quantile(0.99))
        zf_rows.append((other, int(mask.sum())))
    zf_df = pd.DataFrame(zf_rows, columns=["partner", "n_zero_floor"]).sort_values("n_zero_floor", ascending=False)
    best_partner = zf_df.iloc[0]["partner"]
    top_genes_r = cand_means[(means_r[best_partner] == 0) & (cand_means > cand_means.quantile(0.99))] \
        .sort_values(ascending=False).head(20)
    frac_hk = top_genes_r.index.str.match(ribo_hk_pattern).mean() if len(top_genes_r) else np.nan

    return {
        "run": label, "n_probes_requested": len(read_genes(probes_path)), "n_probes_usable": len(usable),
        "broadest_profile": candidate,
        "breadth_ties": ", ".join(breadth.index[breadth.eq(breadth.max())]),
        "largest_dominant_profile": dom.idxmax(),
        "largest_dominant_n": int(dom.max()),
        "largest_dominant_breadth": int(breadth[dom.idxmax()]),
        "candidate_median_proportion": float(props_r[candidate].median()),
        "n_top_boundary_genes": len(top_genes_r),
        "max_breadth": int(breadth.max()),
        "candidate_dominant_n": int(dom.get(candidate, 0)),
        "zero_floor_partner": best_partner,
        "n_zero_floor_genes": int(zf_df.iloc[0]["n_zero_floor"]),
        "frac_top20_housekeeping": round(frac_hk, 2),
    }

rows = [
    diagnose("baseline (344 genes)", stage1_dir, stage2_dir, os.path.join(XDEC_DIR, "stage0", "selected_probes.txt")),
    diagnose("A: paper genes (282)", runs["A_paperGenes"]["stage1_dir"], runs["A_paperGenes"]["stage2_dir"], paper_genes_path),
    diagnose("B: no housekeeping (292)", runs["B_noHK"]["stage1_dir"], runs["B_noHK"]["stage2_dir"], no_hk_probes_path),
]
results_df = pd.DataFrame(rows).set_index("run")
results_df

**Interpretation:** the observed maximum breadth falls from 10 reference types in the baseline to 5 in A and 4 in B. The usable correlation gene counts are 341, 247, and 289 respectively, rather than the requested 344, 282, and 292. This confirms a partial contribution of probe selection to the observed breadth, while improvement in estimation accuracy remains unproven: the correlation gene set also changes, and profiles have not been matched across fits.

Stage 2 refits the full bulk gene matrix using Stage 1 proportions. Its gene universe is unchanged by probe filtering, but its estimates still depend on the resulting proportions. The fraction of prefix-matching genes among the top boundary contrasts is descriptive, not an error rate or a measure of improvement.

**Profile–reference correlation heatmaps for A and B**

Match Figure 2 in [02_visualization.ipynb](02_visualization.ipynb): average reference cells within each (patient, cell type), then compute gene-level Spearman correlations with each estimated profile. Columns follow the same biological-compartment order; rows remain `Profile_1`–`Profile_9`, without clustering. Patient IDs follow the reference export's prefix-before-underscore convention.

Each run uses its own shared probe genes (A: Table S3; B: prefix-filtered probes). Both panels use a fixed blue color scale from 0 to 1. Negative correlations are displayed at the lowest color; the saved correlation tables retain the original signed values. These pseudo-bulk correlations differ from the CLI's mean single-cell correlations used in the summary table. Profile numbers do not imply matched identities across runs.


In [ ]:
# Reuse Figure 2's patient × cell-type reference means and visual layout.
cell_type_order = [
    "pancreatic acinar cell", "pancreatic ductal cell", "enteroendocrine cell",
    "pancreatic stellate cell", "fibroblast", "blood vessel endothelial cell", "neural cell",
    "T cell", "lymphocyte of B lineage", "myeloid leukocyte",
]
if set(cell_type_order) != set(ref_cell_types.unique()):
    raise ValueError("Cell-type order must cover the reference metadata")
cell_palette = dict(zip(cell_type_order, sns.color_palette("tab10", len(cell_type_order))))
patients = pd.Series(refs.columns, index=refs.columns).str.split("_").str[0]
if patients.eq("").any() or not refs.columns.str.contains("_").all():
    raise ValueError("Expected patient-prefixed reference cell IDs")
group_key = patients + " | " + ref_cell_types.astype(str)
reference_means = refs.T.groupby(group_key, sort=True).mean().T
pb_cell_types = pd.Series({c: c.split(" | ", 1)[1] for c in reference_means.columns})
pb_order = pb_cell_types.map({t: i for i, t in enumerate(cell_type_order)}).sort_values(kind="stable").index
reference_means = reference_means.loc[:, pb_order]
profile_order = [f"Profile_{i}" for i in range(1, 10)]
correlation_panels = {}
shared_probe_counts = {}

for label, cfg in runs.items():
    profiles = pd.read_csv(Path(cfg["stage1_dir"]) / "cell_type_profiles.tsv", sep="\t", index_col=0)
    genes = [gene for gene in read_genes(cfg["probes"])
             if gene in refs.index and gene in profiles.index]
    if len(genes) < 3:
        raise ValueError(f"{label}: insufficient shared genes")
    combined = pd.concat([reference_means.loc[genes], profiles.loc[genes, profile_order]], axis=1)
    correlations = combined.corr(method="spearman").loc[profile_order, pb_order]
    if not np.isfinite(correlations.to_numpy()).all():
        raise ValueError(f"{label}: undefined correlations; check for constant expression vectors")
    correlation_panels[label] = correlations
    shared_probe_counts[label] = len(genes)

# Fixed display scale; retain signed values in matrices and TSV exports.
vmin, vmax = 0.0, 1.0
heatmap_dir = Path("paper_duplication/figures")
heatmap_dir.mkdir(parents=True, exist_ok=True)
titles = {"A_paperGenes": "A: extracted Table S3 genes", "B_noHK": "B: prefix-filtered probes"}

for label, correlations in correlation_panels.items():
    grid = sns.clustermap(
        correlations, row_cluster=False, col_cluster=False,
        col_colors=pb_cell_types.loc[pb_order].map(cell_palette).to_numpy(),
        cmap=seq_blue, vmin=vmin, vmax=vmax,
        xticklabels=False, yticklabels=True, figsize=(12, 4.6),
        cbar_kws={"label": "Spearman r", "ticks": [0, 0.25, 0.5, 0.75, 1]}, dendrogram_ratio=(0.02, 0.02),
    )
    grid.ax_heatmap.set_xlabel(f"{len(pb_order)} patient × cell-type reference means (sorted by cell type)")
    grid.ax_heatmap.set_ylabel("Estimated profile (fixed order)")
    grid.ax_heatmap.yaxis.set_label_position("left")
    grid.ax_heatmap.tick_params(axis="y", rotation=0)
    handles = [plt.Rectangle((0, 0), 1, 1, color=cell_palette[t]) for t in cell_type_order]
    legend = grid.ax_heatmap.legend(
        handles, cell_type_order, title="Cell type", bbox_to_anchor=(1.05, 1.02),
        loc="upper left", fontsize=7, title_fontsize=8, frameon=False,
    )
    grid.fig.suptitle(f"{titles[label]}: profile–reference correlation\n"
                      f"{shared_probe_counts[label]} shared genes; color scale 0–1", y=1.04, fontsize=10)
    grid.fig.canvas.draw()
    bbox = legend.get_window_extent(grid.fig.canvas.get_renderer()).transformed(grid.fig.transFigure.inverted())
    grid.cax.set_position([bbox.x0, bbox.y0 - 0.20, 0.015, 0.15])
    grid.cax.yaxis.label.set_fontsize(7)
    output = heatmap_dir / f"fig_{label.lower()}_profile_reference_correlation.png"
    grid.savefig(output, dpi=200, bbox_inches="tight")
    correlations.to_csv(output.with_suffix(".tsv"), sep="\t")
    plt.show()
    plt.close(grid.fig)
    print(f"{titles[label]}: {shared_probe_counts[label]} genes; saved {output}")


### 3. Different reference preprocessing procedures

`03_paper_style_reference_processing.ipynb` compares single-cell and five-cell pseudo-bulk references. The local export has approximately equal cell totals, so original sequencing-depth ranking cannot be recovered; grouping is randomized instead. This is an approximation, not a full reconstruction of the paper's processing.

Run C pretransforms the reference and is unsuitable for judging the intended intervention. Run D passes untransformed pseudo-bulk values to the CLI. In the local code, `normalise_bulk` jointly quantile-normalizes bulk and reference before its per-gene transform; `normalise_refs_independent` separately transforms the reference for correlation labeling. These are distinct paths. Avoid preapplying the transform to either input.

The recorded Run D maximum breadth is 7/10, and its most frequently dominant profile wins in 102/150 samples. This configuration did not lower those descriptive metrics relative to A. It does not rule out reference-processing effects, because grouping, source data, and evaluation references differ.

Across runs, a frequently dominant profile has no reference-type correlation above 0.3. This means low similarity by this metric, not zero correlation. Its identity and malignancy remain unresolved. Assess marker enrichment, alignment across fits, reference coverage, and association with independently measured purity before assigning a tumor label.

### 4. Excluding the single Profile_7-dominant sample (paper's n=149)

The n=182/k=8 initial run had Profile_7 dominant in exactly one sample. The n=150/k=9 panCanAtlas cohort used above has Profile_7 dominant in four samples, a different cohort and k, so the counts are not directly comparable. This section checks whether that one originally-flagged sample is driving the broad reference correlation and boundary-gene signatures, and whether removing it (incidentally reaching the paper's reported n=149) changes them.

In [ ]:
props_k8 = pd.read_csv(os.path.join(XDEC_DIR, "stage1_with_refs_k8", "proportions.tsv"), sep="\t", index_col=0)
dom_k8 = props_k8.idxmax(axis=1)
profile7_dominant_k8 = dom_k8[dom_k8 == "Profile_7"].index
if len(profile7_dominant_k8) != 1:
    raise ValueError(f"Expected exactly one Profile_7-dominant sample in the n=182/k=8 run; found {len(profile7_dominant_k8)}")
artifact_sample = profile7_dominant_k8[0]
print(f"n=182/k=8 Profile_7-dominant sample: {artifact_sample}")

dom_n150 = proportions.idxmax(axis=1)
profile7_dominant_n150 = dom_n150[dom_n150 == "Profile_7"].index
print(f"n=150/k=9 Profile_7-dominant samples ({len(profile7_dominant_n150)}): {list(profile7_dominant_n150)}")
if artifact_sample not in profile7_dominant_n150:
    raise ValueError("Expected the n=182/k=8 artifact sample to remain Profile_7-dominant in the n=150/k=9 cohort")
print(f"\nConfirmed: {artifact_sample} is Profile_7-dominant in both runs; "
      f"it is one of {len(profile7_dominant_n150)} such samples at n=150/k=9.")

Dropping this one sample from the 150-sample panCanAtlas cohort gives exactly 149 samples, matching XDec-CHI's reported n. This is a coincidental numeric match, not a claim that this is the specific sample missing from XDec-CHI's UCSC Xena pull relative to the marker paper's 150-sample cohort — see [sample_filtering.md](sample_filtering.md), which leaves that discrepancy's cause unresolved. The only question tested here is whether removing this specific, independently-flagged outlier changes the Profile_7 pattern.

In [ ]:
n149_bulk_path = os.path.join(XDEC_DIR, "TCGA-PAAD_counts_raw_n149.tsv")
if Path(n149_bulk_path).exists():
    restricted_149 = pd.read_csv(n149_bulk_path, sep="\t", index_col=0)
else:
    if not RUN_PIPELINE:
        raise FileNotFoundError("Missing n=149 cohort subset; set RUN_PIPELINE=True to create it")
    restricted_150 = pd.read_csv(restricted_bulk_path, sep="\t", index_col=0)
    restricted_149 = restricted_150.drop(columns=[artifact_sample])
    restricted_149.to_csv(n149_bulk_path, sep="\t")

if restricted_149.shape[1] != 149 or artifact_sample in restricted_149.columns:
    raise ValueError("Expected the 150-sample cohort minus the one artifact-dominant sample")
if set(restricted_149.columns) != set(restricted_columns) - {artifact_sample}:
    raise ValueError("n=149 cohort must be exactly the n=150 cohort minus the artifact sample")
restricted_149_columns = restricted_149.columns
print(f"Validated {restricted_149.shape[0]} genes × {restricted_149.shape[1]} samples")
del restricted_149

In [ ]:
stage1_dir_149 = os.path.join(XDEC_DIR, "stage1_with_refs_k9_n149")
os.makedirs(stage1_dir_149, exist_ok=True)
proportions_path_149 = os.path.join(stage1_dir_149, "proportions.tsv")

if not all(Path(stage1_dir_149, name).exists() for name in
           ("proportions.tsv", "cell_type_profiles.tsv", "correlation.tsv")):
    result = run_checked(
        [PYTHON_PIXI, XDEC_SCRIPT, "stage1",
         "--bulk", n149_bulk_path,
         "--probes", os.path.join(XDEC_DIR, "stage0", "selected_probes.txt"),
         "--refs", os.path.join(XDEC_DIR, "refs.tsv"),
         "--ref-metadata", os.path.join(XDEC_DIR, "ref_metadata.tsv"),
         "--k", "9",
         "--outdir", stage1_dir_149],
    )

proportions_149 = pd.read_csv(proportions_path_149, sep="\t", index_col=0)
corr_149 = pd.read_csv(os.path.join(stage1_dir_149, "correlation.tsv"), sep="\t", index_col=0)
proportions_149.shape

In [ ]:
stage2_dir_149 = os.path.join(XDEC_DIR, "stage2_with_refs_k9_n149")
os.makedirs(stage2_dir_149, exist_ok=True)
means_path_149 = os.path.join(stage2_dir_149, "stage2_means.tsv")

if not all(Path(stage2_dir_149, name).exists() for name in
           ("stage2_means.tsv", "stage2_stderrs.tsv")):
    result = run_checked(
        [PYTHON_PIXI, XDEC_SCRIPT, "stage2",
         "--bulk-counts", n149_bulk_path,
         "--proportions", proportions_path_149,
         "--outdir", stage2_dir_149],
    )

stage2_means_149 = pd.read_csv(means_path_149, sep="\t", index_col=0)


def validate_outputs_149(props, corr, means):
    for name, frame in {"proportions": props, "correlation": corr, "means": means}.items():
        if not frame.index.is_unique or not frame.columns.is_unique:
            raise ValueError(f"Duplicate labels in {name}")
        if not np.isfinite(frame.to_numpy()).all():
            raise ValueError(f"Nonfinite values in {name}")
    if set(props.columns) != set(corr.columns) or set(props.columns) != set(means.columns):
        raise ValueError("Profile labels disagree between Stage 1/2 outputs")
    if props.shape != (149, 9) or not props.index.equals(restricted_149_columns):
        raise ValueError("Proportions do not match the ordered 149-sample cohort at k=9")
    if (props.to_numpy() < 0).any() or (means.to_numpy() < 0).any():
        raise ValueError("Negative proportions or Stage 2 means")
    if not np.allclose(props.sum(axis=1), 1, atol=1e-5):
        raise ValueError("Proportions do not sum to one")
    if (np.abs(corr.to_numpy()) > 1 + 1e-8).any():
        raise ValueError("Correlation outside [-1, 1]")


validate_outputs_149(proportions_149, corr_149, stage2_means_149)
stage2_means_149.shape

In [ ]:
dominant_counts_149 = proportions_149.idxmax(axis=1).value_counts().reindex(proportions_149.columns, fill_value=0)
broad_corr_149 = (corr_149 > CORR_THRESHOLD).sum(axis=0).sort_values(ascending=False)

comparison = pd.DataFrame({
    "n150_dominant_count": dominant_counts,
    "n149_dominant_count": dominant_counts_149,
    "n150_breadth_r_gt_0.3": broad_corr.reindex(dominant_counts.index),
    "n149_breadth_r_gt_0.3": broad_corr_149.reindex(dominant_counts.index),
})
comparison

In [ ]:
profile7_means_149 = stage2_means_149["Profile_7"]
zero_floor_rows_149 = []
for other in stage2_means_149.columns:
    if other == "Profile_7":
        continue
    other_vals = stage2_means_149[other]
    mask = (other_vals == 0) & (profile7_means_149 > profile7_means_149.quantile(0.99))
    zero_floor_rows_149.append({"partner_profile": other, "n_genes_zero_floor": int(mask.sum())})
zero_floor_df_149 = pd.DataFrame(zero_floor_rows_149).sort_values("n_genes_zero_floor", ascending=False)

zero_floor_comparison = zero_floor_df.set_index("partner_profile")["n_genes_zero_floor"] \
    .to_frame("n150_n_genes_zero_floor") \
    .join(zero_floor_df_149.set_index("partner_profile")["n_genes_zero_floor"].rename("n149_n_genes_zero_floor")) \
    .sort_values("n150_n_genes_zero_floor", ascending=False)
zero_floor_comparison

In [ ]:
dom_149 = proportions_149.idxmax(axis=1)
profile7_dominant_149 = set(dom_149[dom_149 == "Profile_7"].index)
profile7_dominant_150 = set(profile7_dominant_n150)

print(f"n=150 Profile_7-dominant samples: {sorted(profile7_dominant_150)}")
print(f"n=149 Profile_7-dominant samples: {sorted(profile7_dominant_149)}")
print(f"\nRemoving only the artifact sample reproduces the remaining set exactly: "
      f"{(profile7_dominant_150 - {artifact_sample}) == profile7_dominant_149}")

**Interpretation:** removing this one sample does not resolve or meaningfully change the Profile_7 signature. Breadth stays 10/10 reference types at both n=150 and n=149 — identical to the baseline fit. The zero-floor boundary-gene counts against each partner profile are essentially unchanged (within a few genes; partner ranking order is the same set of profiles, with minor reordering among the closest two). The Profile_7-dominant sample set at n=149 is exactly the n=150 set minus the dropped sample — no other sample moved into or out of Profile_7 dominance, so the fit did not reshuffle.

This argues against the one originally-flagged sample being the cause of the artifact: the signature is reproduced with it removed, so it is better read as a property of this fit (this reference/probe combination, k=9, seed) rather than an effect of one outlier sample's presence. It does not establish the converse — that Profile_7 is definitely spurious rather than a real minority compartment — nor does it bear on the paper's actual n=149-vs-150 discrepancy (see the caveat above and [sample_filtering.md](sample_filtering.md)). The unresolved rows in the table in cell 2 (reference composition, initialization/seed, implementation differences) remain the more promising next checks.

### 5. Sliding-window check (XDec-CHI's own method, turned on this question)

XDec-CHI's headline result (e.g. cancer-cell `MSLN` vs. stromal `IFNGR1`) comes from ranking bulk samples by expression of a gene of interest and refitting Stage 1/Stage 2 inside overlapping windows of ≥40 samples. None of Sections 1–4 above used this — each only ever refit the whole n=150/n=149 cohort once per probe/reference/cohort variant.

This section borrows that windowing mechanic to ask a narrower question: if Profile_7 is a genuine minority compartment rather than an NNLS estimation artifact, windows enriched for it should sharpen its reference-correlation signature as enrichment increases; an artifact should look the same regardless of enrichment.

* **Ranking variable:** each sample's own baseline Profile_7 proportion (the profile's natural "gene of interest" analogue here — no single marker gene is assumed a priori)
* **Windows:** 40 samples each (the paper's reported minimum), 6 overlapping windows spanning the full low→high enrichment range of the n=150 cohort
* **Per window:** independently refit Stage 1 (k=9, same probes/refs/seed) and Stage 2, then match the window's own broadest profile back to the baseline Profile_7 profile by Spearman correlation over shared probes — profile numbering is not guaranteed to be stable across independent refits
* This is a diagnostic adaptation, not a reproduction of the paper's gene-gene interaction analysis — it does not attempt to recover `MSLN`/`IFNGR1`-style correlations, only whether Profile_7's own signature depends on enrichment for it


In [ ]:
WINDOW_SIZE = 40
N_WINDOWS = 6

windows_dir = os.path.join(XDEC_DIR, "sliding_window_profile7")
os.makedirs(windows_dir, exist_ok=True)

ranked_samples = proportions["Profile_7"].sort_values(ascending=False).index
starts = np.unique(np.linspace(0, len(ranked_samples) - WINDOW_SIZE, N_WINDOWS).round().astype(int))

baseline_profiles = pd.read_csv(Path(stage1_dir) / "cell_type_profiles.tsv", sep="\t", index_col=0)
baseline_profile7 = baseline_profiles["Profile_7"]

restricted_full = pd.read_csv(restricted_bulk_path, sep="\t", index_col=0)
if not restricted_full.columns.equals(proportions.index):
    raise ValueError("Cached n=150 bulk and baseline proportions are not sample-aligned")


def match_to_baseline_profile7(window_profiles, genes):
    common = [g for g in genes if g in baseline_profile7.index and g in window_profiles.index]
    if len(common) < 10:
        raise ValueError("Too few shared probes to match a window profile to baseline Profile_7")
    corrs = window_profiles.loc[common].corrwith(baseline_profile7.loc[common], method="spearman")
    if corrs.isna().all():
        raise ValueError("All window-profile correlations to baseline Profile_7 are undefined")
    return corrs.idxmax(), float(corrs.max())


window_rows = []
for start in starts:
    end = int(start) + WINDOW_SIZE
    window_samples = ranked_samples[start:end]
    tag = f"rank{start:03d}_{end:03d}"
    bulk_path = os.path.join(windows_dir, f"bulk_{tag}.tsv")
    s1_dir = os.path.join(windows_dir, f"stage1_{tag}")
    s2_dir = os.path.join(windows_dir, f"stage2_{tag}")
    os.makedirs(s1_dir, exist_ok=True)
    os.makedirs(s2_dir, exist_ok=True)

    if Path(bulk_path).exists():
        window_bulk_cols = pd.read_csv(bulk_path, sep="\t", index_col=0, nrows=0).columns
        if list(window_bulk_cols) != list(window_samples):
            raise ValueError(f"Cached window bulk columns disagree with the current ranking for {tag}")
    else:
        if not RUN_PIPELINE:
            raise FileNotFoundError(f"Missing window bulk subset for {tag}; set RUN_PIPELINE=True to create it")
        restricted_full.loc[:, window_samples].to_csv(bulk_path, sep="\t")

    if not all(Path(s1_dir, name).exists() for name in
               ("proportions.tsv", "cell_type_profiles.tsv", "correlation.tsv")):
        run_checked([PYTHON_PIXI, XDEC_SCRIPT, "stage1",
                     "--bulk", bulk_path,
                     "--probes", os.path.join(XDEC_DIR, "stage0", "selected_probes.txt"),
                     "--refs", os.path.join(XDEC_DIR, "refs.tsv"),
                     "--ref-metadata", os.path.join(XDEC_DIR, "ref_metadata.tsv"),
                     "--k", "9",
                     "--outdir", s1_dir])

    w_props = pd.read_csv(os.path.join(s1_dir, "proportions.tsv"), sep="\t", index_col=0)
    w_profiles = pd.read_csv(os.path.join(s1_dir, "cell_type_profiles.tsv"), sep="\t", index_col=0)
    w_corr = pd.read_csv(os.path.join(s1_dir, "correlation.tsv"), sep="\t", index_col=0)
    if w_props.shape != (WINDOW_SIZE, 9) or not w_props.index.equals(window_samples):
        raise ValueError(f"{tag}: Stage 1 output does not match the windowed sample set")

    if not all(Path(s2_dir, name).exists() for name in ("stage2_means.tsv", "stage2_stderrs.tsv")):
        run_checked([PYTHON_PIXI, XDEC_SCRIPT, "stage2",
                     "--bulk-counts", bulk_path,
                     "--proportions", os.path.join(s1_dir, "proportions.tsv"),
                     "--outdir", s2_dir])
    w_means = pd.read_csv(os.path.join(s2_dir, "stage2_means.tsv"), sep="\t", index_col=0)

    matched_profile, match_r = match_to_baseline_profile7(w_profiles, probe_list)
    ref_corrs = w_corr[matched_profile].sort_values(ascending=False)
    matched_means = w_means[matched_profile]
    zf_counts = {
        other: int(((w_means[other] == 0) & (matched_means > matched_means.quantile(0.99))).sum())
        for other in w_means.columns if other != matched_profile
    }
    window_rows.append({
        "window": tag,
        "mean_profile7_proportion": float(proportions.loc[window_samples, "Profile_7"].mean()),
        "matched_profile": matched_profile,
        "match_to_baseline_profile7_r": match_r,
        "breadth_r_gt_0.3": int((w_corr[matched_profile] > CORR_THRESHOLD).sum()),
        "top_reference_r": float(ref_corrs.iloc[0]),
        "second_reference_r": float(ref_corrs.iloc[1]),
        "reference_margin": float(ref_corrs.iloc[0] - ref_corrs.iloc[1]),
        "max_n_zero_floor_genes": max(zf_counts.values()),
    })

window_df = pd.DataFrame(window_rows).sort_values("mean_profile7_proportion").reset_index(drop=True)
window_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
axes[0].plot(window_df["mean_profile7_proportion"], window_df["breadth_r_gt_0.3"], "o-", color="#2a78d6")
axes[0].set_xlabel("window mean Profile_7 proportion")
axes[0].set_ylabel("breadth (ref. types, r>0.3)")
axes[0].set_ylim(-0.5, 10.5)

axes[1].plot(window_df["mean_profile7_proportion"], window_df["reference_margin"], "o-", color="#1c5cab")
axes[1].set_xlabel("window mean Profile_7 proportion")
axes[1].set_ylabel("top − 2nd reference r (margin)")

fig.suptitle("Matched-profile signature vs. window enrichment for Profile_7", y=1.03, fontsize=10)
fig.tight_layout()
window_fig_path = Path("paper_duplication/figures") / "fig_sliding_window_profile7.png"
window_fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(window_fig_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print("saved", window_fig_path)